In [2]:
# M6.2 - Validating Joins

# just because a join runs without error doesnt mean it is correct
# issues like:
# duplicate keys - one PO matching multiple supplier records can multiply rows
# missing matches - rows mistakenly dropped from an inner join
# cardinality violations - where 1:M relationships are treated as 1:1

# Pandas allows you to catch these issues before they cause reporting errors

# import libraries
import pandas as pd
import numpy as np

# create the data frames - PO + Supplier Master
df_po = pd.DataFrame({
    "po_id":    ["PO-001", "PO-002", "PO-003", "PO-004"],
    "supplier": ["Acme", "GlobalCo", "Acme", "GlobalCo"],
    "amount":   [1500, 2200, 800, 1200]
})

# supplier master — Acme appears twice (duplicate key problem)
df_suppliers = pd.DataFrame({
    "supplier": ["Acme", "Acme", "GlobalCo", "FastParts"],
    "country":  ["USA", "Canada", "UK", "Germany"],
    "rating":   [4.5, 4.2, 3.8, 4.2]
})

print(df_po)
print(df_suppliers)

    po_id  supplier  amount
0  PO-001      Acme    1500
1  PO-002  GlobalCo    2200
2  PO-003      Acme     800
3  PO-004  GlobalCo    1200
    supplier  country  rating
0       Acme      USA     4.5
1       Acme   Canada     4.2
2   GlobalCo       UK     3.8
3  FastParts  Germany     4.2


In [3]:
# Exercise 2
# Check for duplicates in your key columns before you join
print(df_suppliers["supplier"].duplicated())
print(df_suppliers["supplier"].duplicated().sum())

# value_counts whows what keys are duplicated
print(df_suppliers["supplier"].value_counts())

0    False
1     True
2    False
3    False
Name: supplier, dtype: bool
1
supplier
Acme         2
GlobalCo     1
FastParts    1
Name: count, dtype: int64


In [ ]:
# Exercise 3: The Vaidate parameter

# .merge has a validate paramter that raises an error if the join doesnt match the expected cardinality

# cardinality
# "one_to_one" — each key appears once in both DataFrames
# "one_to_many" — each key appears once in left, multiple times in right
# "many_to_one" — each key appears multiple times in left, once in right
# "many_to_many" — keys can appear multiple times in both

# .merge examples with errors
# this example shows the wrong validation
# since suppliers appers twice in df_suppliers the validaton should equal "many_to_many"
try:
    df_po.merge(right=df_suppliers, on="supplier", how="left", validate="many_to_one")
except Exception as e:
    print(f"Validation error: {e}")

# .merge example without error
# the supplier column has duplicates in both dataframes
# validate="many_to_many"
result = df_po.merge(right=df_suppliers, on="supplier", how="left", validate="many_to_many")
print(result)
print(f"Expected 4 rows, got {len(result)} rows")

Validation error: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 supplier
    Acme ...
    po_id  supplier  amount country  rating
0  PO-001      Acme    1500     USA     4.5
1  PO-001      Acme    1500  Canada     4.2
2  PO-002  GlobalCo    2200      UK     3.8
3  PO-003      Acme     800     USA     4.5
4  PO-003      Acme     800  Canada     4.2
5  PO-004  GlobalCo    1200      UK     3.8
Expected 4 rows, got 6 rows
